# 人工标记GNSS删除（仅本次授权）
写入日期：2026-10-21  
修改日期：2026-10-22

只处理用户在“冲突轨迹与GNSS.md”的GNSS行数后加“-”的城市访问，共55次访问、92条原始作业段（2022年15行、2023年77行）。没有“-”的行保留，包括一处行数留空的无锡市访问，不补写用户文件。先核对原文件及标记，重建连续同城访问，保存删除记录，再只删对应原始记录。保留行的字节、先后顺序、表头、换行和字段均不改变。

本次例外只允许删除这些行，不允许删除其他疑似错误、空值或重复记录。普通运行只检查；本次执行须设置GNSS_APPLY_MARKED_DELETION=1。已有完成记录时仅核对，不再删除。原GNSS由用户另有备份，本Notebook不复制全表。旧生成结果已统一保存到Archive/2026-10-21_人工删除GNSS前。第二步不运行。


## 参数


In [1]:
from pathlib import Path
import csv
import hashlib
import os
import re
import tempfile
import pandas as pd
import duckdb
from IPython.display import display

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'README.md').is_file() and (p/'Data').is_dir())
result_dir = project_root/'Results/00_数据审计'
manifest_path = project_root/'Data/data_manifest.csv'
annotation_path = result_dir/'冲突轨迹与GNSS.md'
notebook_relative = 'Code/01_轨迹转城市接续/delete_marked_gnss.ipynb'
archive_relative = 'Archive/2026-10-21_人工删除GNSS前'
detail_path = result_dir/'GNSS删除明细.csv'
check_path = result_dir/'GNSS删除检查.csv'
visit_path = result_dir/'人工删除城市.csv'
apply_deletion = os.getenv('GNSS_APPLY_MARKED_DELETION','0') == '1'
expected_annotation_hash = '74ef095e7dd113837ad8de72a997a0fa7e55d3bd6ac9ec978321cca4ca44aeef'
original_hashes = {
    2021: '990dc1a3b4406dcd23511d6df83a1925ce60c3fd9f034fce31dde00d93c602f8',
    2022: 'bdd6df39cd79c67e7c4ff96e0eaa03db6eb6309761cd46c6856bb4d6c3ccd9ad',
    2023: 'e8668e98a55d88f1e956c0750d1baa278ae97f646e20088a5b2c3539f84e06d7'}
original_counts = {2021:770845,2022:2475805,2023:3976426}
sort_sql = 'date_value,start_ts,end_ts,center_ts NULLS LAST,info_id NULLS LAST,id NULLS LAST,source_row_no'

def fingerprint(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle,'sha256').hexdigest()

raw_paths = {year:project_root/f'Data/00_原始数据_只读/final_filtered_{str(year)[-2:]}_modified.csv' for year in original_hashes}
manifest_table = pd.read_csv(manifest_path,dtype='string',keep_default_na=False,encoding='utf-8-sig')
completed = check_path.exists()
if completed:
    checks = pd.read_csv(check_path,dtype='string',keep_default_na=False,encoding='utf-8-sig')
    assert len(checks)==3 and checks['状态'].eq('已完成').all(), '上次删除尚未完整完成，请先人工核对，禁止重复删除。'
    for year,path in raw_paths.items():
        row = checks.loc[checks['年份'].eq(str(year))].iloc[0]
        assert fingerprint(path)==row['删除后文件指纹'], f'{year}年GNSS已再次变化，停止。'
    assert detail_path.is_file() and visit_path.is_file()
    print('本次授权删除已经完成；以下只核对已有结果，不再次修改GNSS。')
else:
    assert not detail_path.exists() and not visit_path.exists(), '发现未完成的删除记录，先人工核对。'


本次授权删除已经完成；以下只核对已有结果，不再次修改GNSS。


## 读取


In [2]:
if not completed:
    assert fingerprint(annotation_path)==expected_annotation_hash, '人工标记文件又变了，停止，不自动选择新版本。'
    annotations = []
    for section in re.split(r'(?=^## )',annotation_path.read_text(encoding='utf-8'),flags=re.M):
        if not section.strip():
            continue
        heading = re.match(r'## (\d{4})年 · did=(.*?) · 链段(\d+)：(.*)',section)
        assert heading, '路径标题格式不能识别。'
        year,did,chain,city_path = heading.groups()
        assert chain=='1'
        lines = [line for line in section.splitlines() if line.startswith('| ') and not line.startswith('| did ') and not line.startswith('| ---')]
        for visit_order,line in enumerate(lines,1):
            values = [cell.strip() for cell in line.strip('|').split('|')]
            assert len(values)==6 and values[:2]==[did,chain]
            count_text = values[5]
            assert re.fullmatch(r'\d+-?',count_text) or count_text==''
            annotations.append({'年份':int(year),'农机编号':did,'链段编号':1,'原访问序号':visit_order,
                '城市':values[2],'最早进入时间':values[3],'最晚离开时间':values[4],
                '标记原文':count_text,'删除标记':count_text.endswith('-'),'完整城市路径':city_path})
    annotation_table = pd.DataFrame(annotations)
    assert len(annotation_table)==183 and annotation_table[['年份','农机编号']].drop_duplicates().shape[0]==22
    assert annotation_table['删除标记'].sum()==55
    empty_counts = annotation_table.loc[annotation_table['标记原文'].eq('')]
    assert len(empty_counts)==1 and empty_counts.iloc[0]['农机编号']=='102012100391' and empty_counts.iloc[0]['原访问序号']==2
    assert not empty_counts['删除标记'].any(), '空白没有删除标记，必须保留。'
    for year,path in raw_paths.items():
        relative = path.relative_to(project_root).as_posix()
        item = manifest_table.loc[manifest_table['relative_path'].eq(relative)]
        assert len(item)==1 and item.iloc[0]['formal_status']=='正式'
        assert item.iloc[0]['sha256']==original_hashes[year] and fingerprint(path)==original_hashes[year]
        assert int(item.iloc[0]['row_count'])==original_counts[year]
    print('人工标记、三年GNSS正式版本及原始文件指纹已核对。')


## 处理


In [3]:
if not completed:
    # 先给完整原文件编号，再选对应农机；不按时间区间或首末行号粗略截取。
    member_parts = []
    connection = duckdb.connect()
    connection.execute('SET preserve_insertion_order=true')
    try:
        for year,group in annotation_table.groupby('年份',sort=True):
            connection.register('wanted',pd.DataFrame({'did':group['农机编号'].unique()}))
            connection.execute("""
                CREATE OR REPLACE TEMP TABLE selected_raw AS
                SELECT * FROM (
                    SELECT row_number() OVER ()::BIGINT AS source_row_no,*
                    FROM read_csv(?,header=true,all_varchar=true,parallel=false,ignore_errors=false))
                WHERE trim(did) IN (SELECT did FROM wanted)
            """,[str(raw_paths[year])])
            connection.execute("""
                CREATE OR REPLACE TEMP TABLE parsed AS
                SELECT source_row_no,NULLIF(trim(info_id),'') AS info_id,NULLIF(trim(id),'') AS id,
                    trim(did) AS machine_id,CAST(trim(date) AS DATE) AS date_value,
                    CAST(trim(start_time) AS TIMESTAMP) AS start_ts,CAST(trim(end_time) AS TIMESTAMP) AS end_ts,
                    COALESCE(try_strptime(trim(center_time),'%Y%m%d%H%M%S'),
                             try_cast(trim(center_time) AS TIMESTAMP)) AS center_ts,
                    NULLIF(trim(center_location),'') AS city
                FROM selected_raw
            """)
            assert connection.execute('SELECT count(*) FROM parsed WHERE city IS NULL OR machine_id IS NULL OR start_ts IS NULL OR end_ts IS NULL OR end_ts<start_ts').fetchone()[0]==0
            members = connection.execute(f"""
                WITH flags AS (
                    SELECT *,CASE WHEN city IS DISTINCT FROM lag(city) OVER(PARTITION BY machine_id ORDER BY {sort_sql})
                        THEN 1 ELSE 0 END AS new_visit FROM parsed),
                numbered AS (
                    SELECT *,sum(new_visit) OVER(PARTITION BY machine_id ORDER BY {sort_sql} ROWS UNBOUNDED PRECEDING)::BIGINT AS visit_order
                    FROM flags)
                SELECT n.source_row_no,n.machine_id,n.visit_order,n.city,n.start_ts,n.end_ts,r.*
                    EXCLUDE(source_row_no)
                FROM numbered n JOIN selected_raw r USING(source_row_no)
                ORDER BY n.machine_id,n.visit_order,n.source_row_no
            """).df().fillna('')
            for did,track in members.groupby('machine_id',sort=False):
                expected = group.loc[group['农机编号'].eq(did)].sort_values('原访问序号')
                visits = track.groupby('visit_order',sort=True).agg(city=('city','first'),entry=('start_ts','min'),exit=('end_ts','max'),count=('source_row_no','size'))
                assert len(visits)==len(expected) and '→'.join(visits['city'])==expected.iloc[0]['完整城市路径']
                for _,mark in expected.iterrows():
                    visit = visits.loc[mark['原访问序号']]
                    assert (visit['city'],str(visit['entry']),str(visit['exit']))==(mark['城市'],mark['最早进入时间'],mark['最晚离开时间'])
                    if mark['标记原文']:
                        assert int(mark['标记原文'].rstrip('-'))==int(visit['count']), '标记行数与重建的城市访问不符。'
            marked = group.loc[group['删除标记']].rename(columns={'农机编号':'machine_id','原访问序号':'visit_order'})
            chosen = members.merge(marked[['machine_id','visit_order','标记原文']],on=['machine_id','visit_order'],validate='many_to_one')
            chosen.insert(0,'年份',year)
            member_parts.append(chosen)
    finally:
        connection.close()
    selected_rows = pd.concat(member_parts,ignore_index=True)
    assert len(selected_rows)==92 and selected_rows.groupby('年份').size().to_dict()=={2022:15,2023:77}
    assert not selected_rows[['年份','source_row_no']].duplicated().any()
    selected_visits = annotation_table.loc[annotation_table['删除标记']].copy()
    selected_visits['GNSS行数'] = selected_visits['标记原文'].str.rstrip('-').astype(int)
    assert selected_visits['GNSS行数'].sum()==92
    display(selected_visits.groupby('年份',as_index=False).agg(删除城市访问数=('GNSS行数','size'),删除GNSS行数=('GNSS行数','sum')))


## 检查


In [4]:
if not completed:
    # csv.reader每次读取一条逻辑记录，同时保留它原有的字节，兼容带引号和跨行字段。
    staged = {}
    check_records = []
    deletion_records = []
    for year,path in raw_paths.items():
        chosen = selected_rows.loc[selected_rows['年份'].eq(year)].set_index('source_row_no')
        delete_numbers = set(chosen.index.astype(int))
        before_hash = hashlib.sha256()
        kept_hash = hashlib.sha256()
        pending_bytes = []
        raw_columns = []
        stage_path = None
        output = None
        if delete_numbers:
            descriptor,temporary = tempfile.mkstemp(prefix='gnss_marked_',suffix='.csv',dir=path.parent)
            os.close(descriptor)
            stage_path = Path(temporary)
            output = stage_path.open('wb')
        try:
            with path.open('rb') as source:
                def decoded_lines():
                    for line in source:
                        pending_bytes.append(line)
                        before_hash.update(line)
                        yield line.decode('utf-8') # 不改原字节；BOM只在表头解析时去掉。
                reader = csv.reader(decoded_lines())
                raw_columns = next(reader)
                raw_columns[0] = raw_columns[0].lstrip('\ufeff')
                assert len(raw_columns)==26 and len(set(raw_columns))==26
                header_bytes = b''.join(pending_bytes); pending_bytes.clear()
                kept_hash.update(header_bytes)
                if output: output.write(header_bytes)
                row_count = deleted_count = 0
                for row_count,values in enumerate(reader,1):
                    record_bytes = b''.join(pending_bytes); pending_bytes.clear()
                    assert len(values)==len(raw_columns), (year,row_count)
                    if row_count in delete_numbers:
                        expected = chosen.loc[row_count]
                        assert values==[str(expected[column]) for column in raw_columns], f'{year}年第{row_count}条原值不符'
                        deletion_records.append({
                            '年份':year,'来源文件':path.relative_to(project_root).as_posix(),
                            '删除前源行号':row_count,'原访问序号':int(expected['visit_order']),
                            '标记原文':expected['标记原文'],'删除原因':'用户在城市访问GNSS行数后加-，删除该次访问全部原始行',
                            **dict(zip(raw_columns,values))})
                        deleted_count+=1
                    else:
                        kept_hash.update(record_bytes)
                        if output: output.write(record_bytes)
                assert row_count==original_counts[year] and deleted_count==len(delete_numbers)
            if output:
                output.flush(); os.fsync(output.fileno()); output.close(); output=None
                assert fingerprint(stage_path)==kept_hash.hexdigest()
                staged[year]=stage_path
            assert before_hash.hexdigest()==original_hashes[year], '处理期间原始文件发生变化'
            check_records.append({'年份':year,'删除前行数':row_count,'删除行数':deleted_count,
                '删除后行数':row_count-deleted_count,'删除前文件指纹':before_hash.hexdigest(),
                '删除后文件指纹':kept_hash.hexdigest(),'标记文件指纹':expected_annotation_hash,
                '保留内容检查':'剩余记录字节与原顺序均未改','状态':'待执行'})
        finally:
            if output: output.close()
    assert len(deletion_records)==92
    checks = pd.DataFrame(check_records)
    display(checks[['年份','删除前行数','删除行数','删除后行数','状态']])
    if not apply_deletion:
        # 预览只检查，不留下可能被误用的中间文件。
        for temporary in staged.values(): temporary.unlink()
        print('仅检查，GNSS未修改；本次授权执行时设置GNSS_APPLY_MARKED_DELETION=1。')


## 输出


In [5]:
if not completed and apply_deletion:
    # 所有年份已通过检查，先保存待执行记录，失败时也可知道哪些文件曾经准备替换。
    pd.DataFrame(deletion_records).to_csv(detail_path,index=False,encoding='utf-8-sig')
    selected_visits.to_csv(visit_path,index=False,encoding='utf-8-sig')
    checks.to_csv(check_path,index=False,encoding='utf-8-sig')
    for year,temporary in staged.items():
        assert fingerprint(raw_paths[year])==original_hashes[year], '替换前原文件已变化'
        os.replace(temporary,raw_paths[year])
    for _,row in checks.iterrows():
        assert fingerprint(raw_paths[int(row['年份'])])==row['删除后文件指纹']
    checks['状态']='已完成'
    checks.to_csv(check_path,index=False,encoding='utf-8-sig')
    for _,row in checks.iterrows():
        year=int(row['年份']); relative=raw_paths[year].relative_to(project_root).as_posix()
        mask=manifest_table['relative_path'].eq(relative)
        manifest_table.loc[mask,'row_count']=str(row['删除后行数'])
        manifest_table.loc[mask,'file_size_bytes']=str(raw_paths[year].stat().st_size)
        manifest_table.loc[mask,'sha256']=row['删除后文件指纹']
        manifest_table.loc[mask,'notes']=f'原始文件继续只读；2026-10-21仅本次授权删除人工标记{row["删除行数"]}行；删除依据和前后检查见GNSS删除检查.csv'
    for identifier,path,grain,count in [
        ('GNSS_MARKED_DELETION',detail_path,'一行一条本次从GNSS删除的完整原始记录',92),
        ('GNSS_MARKED_VISITS',visit_path,'一行一次用户标记删除的连续同城访问',55),
        ('GNSS_MARKED_CHECK',check_path,'一行一个年份的删除前后检查',3)]:
        entry={column:'' for column in manifest_table.columns}
        entry.update(file_id=identifier,relative_path=path.relative_to(project_root).as_posix(),file_name=path.name,
            file_role='quality_output',formal_status='正式',read_only='false',year='2021-2023',
            time_range='各年05-01至06-25',data_grain=grain,row_count=str(count),
            file_size_bytes=str(path.stat().st_size),sha256=fingerprint(path),encoding='UTF-8-SIG',
            source_or_generator=notebook_relative,upstream='三份删除前GNSS；用户手工标记的冲突轨迹与GNSS.md',
            purpose='追溯本次一次性授权删除，不作研究样本输入',
            notes=f'标记原件及旧生成结果见{archive_relative}；原始GNSS全表未另备份；旧行号只对应删除前版本')
        manifest_table=pd.concat([manifest_table.loc[~manifest_table['file_id'].eq(identifier)],pd.DataFrame([entry])],ignore_index=True)
    # 只在这一次实际删除时使旧下游结果退出正式范围；今后仅核对，不再次改清单。
    stale_mask = (manifest_table['source_or_generator'].str.startswith('Code/02_构建链路数据/')
                  | manifest_table['source_or_generator'].eq('Code/01_轨迹转城市接续/analyze_succession_days.ipynb'))
    manifest_table.loc[stale_mask,'formal_status'] = '历史结果'
    manifest_table.loc[stale_mask,'notes'] = 'GNSS已按人工标记更新；本次按用户要求未重跑第二步及其间隔统计；旧结果仅供追溯，禁止与新版第一步混用'
    manifest_table.to_csv(manifest_path,index=False,encoding='utf-8-sig')
    completed=True
    print('本次授权删除完成：2021年0行、2022年15行、2023年77行。GNSS原文件不再允许本Notebook重复删除。')
if completed:
    display(pd.read_csv(check_path,encoding='utf-8-sig')[['年份','删除前行数','删除行数','删除后行数','状态']])


,年份,删除前行数,删除行数,删除后行数,状态
0,2021,770845,0,770845,已完成
1,2022,2475805,15,2475790,已完成
2,2023,3976426,77,3976349,已完成
